# MAIW v2 Setup — Warehouse World Generator

This notebook guides you through creating your own reproducible MAIW warehouse world.

**What you'll do:**
1. Check your environment
2. Configure your warehouse
3. Generate the Warehouse World (WarehouseDataPack)
4. Inspect the Operational Graph
5. Validate the DataPack
6. Verify reproducibility
7. Preview available scenarios
8. Launch MAIW Demo Mode

**Architecture:**
```
WarehouseWorldConfig
→ WarehouseWorldGenerator (deterministic, seed-based)
→ Canonical Operational Graph
→ WarehouseDataPack (immutable, verifiable)
→ ScenarioOverlay (disruption events)
→ DemoWarehouseWorld (mutable runtime)
→ Simulation Providers → WarehouseState → Agents
```

> **Reproducibility:** Same config + same seed = same warehouse world.  
> Two developers generating from the same parameters get an identical semantic graph.

In [ ]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path().resolve()
print(f"Repository:  {REPO_ROOT}")
print(f"Python:      {sys.version.split()[0]}")
print()

# Check packages
for pkg in ["maiw_world", "pydantic", "yaml"]:
    try:
        __import__(pkg)
        print(f"  {pkg:20s} OK")
    except ImportError:
        print(f"  {pkg:20s} MISSING — run: pip install -e packages/maiw-world")

print()

# Check NVIDIA_API_KEY (mask it)
api_key = os.environ.get("NVIDIA_API_KEY", "")
if api_key:
    print(f"  NVIDIA_API_KEY       CONFIGURED ({api_key[:8]}...)")
else:
    print(f"  NVIDIA_API_KEY       NOT CONFIGURED")
    print("  Set it in your shell: export NVIDIA_API_KEY=nvapi-your-key-here")

## Cell 03 — Warehouse Configuration

Choose a configuration preset. The canonical DC-47 preset is the standard demo/evaluation world.
Use `small()` for fast local iteration (< 1s generation, 100 SKUs).

In [ ]:
from maiw_world.config import WarehouseWorldConfig

# Option A: Use canonical DC-47 preset
config = WarehouseWorldConfig.dc47_demo()

# Option B: Load from YAML (edit data/world-configs/dc47-demo.yaml)
# config = WarehouseWorldConfig.from_yaml("data/world-configs/dc47-demo.yaml")

# Option C: Use small preset for fast local validation
# config = WarehouseWorldConfig.small()

print("Warehouse Configuration")
print("=" * 40)
print(f"  Warehouse ID   {config.warehouse_id}")
print(f"  Dataset ID     {config.dataset_id}")
print(f"  Seed           {config.seed}")
print(f"  Zones          {config.facility.zone_count}")
print(f"  Locations      {config.facility.location_count}")
print(f"  Workers/shift  {config.labor.workers_per_shift}  ({config.labor.shift_count} shifts)")
print(f"  AGVs           {config.equipment.agv_count}")
print(f"  Forklifts      {config.equipment.forklift_count}")
print(f"  SKUs           {config.inventory.sku_count:,}")
print(f"  Orders/day     {config.orders.daily_order_count:,}")
print(f"  Active waves   {config.waves.active_wave_count}")
print(f"  Tasks          {config.waves.task_count}")
print(f"  History days   {config.history.history_days}")

In [ ]:
import time
from maiw_world.generator import WarehouseWorldGenerator

print("Generating Warehouse World...")
print("(This takes 1–5 seconds for DC-47, under 1s for small preset)\n")

t0 = time.time()
result = WarehouseWorldGenerator(config).generate()
elapsed = time.time() - t0

g = result.graph
r = result.report

print("WAREHOUSE GENERATED\n")
print(f"  Warehouse        {config.warehouse_id}")
print(f"  Dataset          {config.dataset_id}")
print(f"  Seed             {config.seed}")
print()
print(f"  Warehouses       {r.entity_counts.warehouses}")
print(f"  Zones            {r.entity_counts.zones}")
print(f"  Locations        {r.entity_counts.locations:,}")
print(f"  Workers          {r.entity_counts.workers:,}")
print(f"  Equipment        {r.entity_counts.equipment}")
print(f"  SKUs             {r.entity_counts.skus:,}")
print(f"  Inventory pos.   {r.entity_counts.inventory_positions:,}")
print(f"  Orders           {r.entity_counts.orders:,}")
print(f"  Waves            {r.entity_counts.waves}")
print(f"  Tasks            {r.entity_counts.tasks:,}")
print()
print(f"  Edges            {r.edge_count:,}")
print(f"  History events   {g.event_count:,}")
print()
print(f"  Validation       {'PASS' if r.validation_result.passed else 'FAIL'}")
print(f"  Duration         {elapsed*1000:.0f} ms")

In [ ]:
from maiw_world.datapack import WarehouseDataPack, compute_semantic_checksum
from pathlib import Path

PACK_DIR = Path(f"data/worlds/{config.dataset_id}")
OVERWRITE = False  # change to True to regenerate an existing pack

if PACK_DIR.exists():
    print(f"DataPack already exists: {PACK_DIR}")
    print("Validating existing pack...")
    v = WarehouseDataPack.verify(PACK_DIR)
    if v.passed:
        print("  Validation: PASS — existing pack is intact, reusing.")
        existing_manifest = WarehouseDataPack.read_manifest(PACK_DIR)
        existing_cs = existing_manifest["semantic_checksum"]
        print(f"  Checksum: {existing_cs[:16]}...")
    else:
        print(f"  Validation: FAIL — {v.errors}")
        print("  Set OVERWRITE = True in this cell to regenerate.")
else:
    OVERWRITE = True

if OVERWRITE or not PACK_DIR.exists():
    print(f"\nWriting DataPack to: {PACK_DIR}")
    WarehouseDataPack.write(g, config, PACK_DIR)
    cs = compute_semantic_checksum(g)
    print(f"\nDataPack written.")
    print(f"  Path       {PACK_DIR}")
    print(f"  Checksum   {cs[:16]}...")
    print(f"\nThis checksum is stable: same config + seed → same checksum, always.")

In [ ]:
from maiw_world.entities import EntityType
from maiw_world.edges import RelationshipType

print("Operational Graph Summary")
print("=" * 40)
summary = g.summary()
for key, val in sorted(summary.items()):
    print(f"  {key:30s} {val:,}")

# Inspect one wave's neighborhood
waves = sorted(g.entities_by_type(EntityType.WAVE), key=lambda w: w.id)
if waves:
    wave = waves[0]
    print(f"\nSample Wave: {wave.id} (wave #{wave.wave_number}, status={wave.status})")

    # Tasks
    task_edges = g.incoming_edges(wave.id, RelationshipType.BELONGS_TO)
    print(f"\n  Tasks ({len(task_edges)}):")
    for e in task_edges[:5]:
        task = g.get_entity(e.source_id)
        if task:
            print(f"    {task.id}  {task.task_type.value:12s}  {task.status.value}")
    if len(task_edges) > 5:
        print(f"    ... and {len(task_edges)-5} more")

    # Orders fulfilled
    order_edges = g.outgoing_edges(wave.id, RelationshipType.FULFILLS)
    print(f"\n  Fulfills {len(order_edges)} order(s)")

    # Cutoffs
    cutoff_edges = g.outgoing_edges(wave.id, RelationshipType.CONSTRAINED_BY)
    for e in cutoff_edges:
        cutoff = g.get_entity(e.target_id)
        if cutoff:
            print(f"\n  Constrained by: {cutoff.carrier} cutoff at {cutoff.cutoff_time}")

In [ ]:
print("Validating DataPack...")
vresult = WarehouseDataPack.verify(PACK_DIR)

print(f"\n  DataPack        {PACK_DIR}")
print(f"  Dataset         {config.dataset_id}")
print(f"  Warehouse       {config.warehouse_id}")
print(f"  Seed            {config.seed}")
print()
print(f"  Manifest        {'PASS' if vresult.manifest_valid else 'FAIL'}")
print(f"  File checksums  {'PASS' if vresult.file_checksums_match else 'FAIL'}")
print(f"  Semantic check  {'PASS' if vresult.semantic_checksum_match else 'FAIL'}")
gv = vresult.graph_validation
print(f"  Graph validity  {'PASS' if (gv is not None and gv.passed) else 'FAIL'}")
print()
print(f"  Status          {'PASS' if vresult.passed else 'FAIL'}")

if not vresult.passed:
    print("\nErrors:")
    for e in vresult.errors:
        print(f"  - {e}")
    raise RuntimeError("DataPack validation failed. Fix errors above before continuing.")

In [ ]:
# Demonstrate: same config + seed = same checksum
from maiw_world.datapack import compute_semantic_checksum

cs1 = compute_semantic_checksum(g)
g2 = WarehouseWorldGenerator(config).generate().graph
cs2 = compute_semantic_checksum(g2)

print("Reproducibility Check")
print("=" * 40)
print(f"  Run 1 checksum: {cs1[:32]}...")
print(f"  Run 2 checksum: {cs2[:32]}...")
print(f"  Match: {'YES' if cs1 == cs2 else 'NO — UNEXPECTED DIFFERENCE'}")
print()
print("Same config + seed = identical warehouse world, every time.")

In [ ]:
from maiw_world.datapack import WarehouseDataPack
from maiw_world.scenario import labor_constraint_scenario, equipment_failure_scenario

loaded_graph = WarehouseDataPack.load(PACK_DIR)

print("Available Scenarios")
print("=" * 40)

# Registry from world_loader (shown inline for notebook clarity)
scenario_registry = {
    "labor_constraint_wave_risk": "DataPack-native",
    "equipment_failure":          "DataPack-native",
    "healthy_baseline":           "DataPack-native",
    "stale_state":                "Compatibility adapter",
    "state_drift":                "Compatibility adapter",
}

for name, status in scenario_registry.items():
    print(f"  {name:35s}  {status}")

# Preview labor constraint scenario
overlay = labor_constraint_scenario(loaded_graph)
print(f"\nScenario preview: {overlay.name}")
print(f"  Events:  {len(overlay.events)}")
print(f"  Tags:    {', '.join(overlay.tags)}")
print(f"\n  Timeline:")
for ev in sorted(overlay.events, key=lambda e: e.sim_time_offset_seconds)[:6]:
    print(f"    t+{ev.sim_time_offset_seconds:6.0f}s  {ev.kind.value:28s}  {ev.label[:50]}")
if len(overlay.events) > 6:
    print(f"    ... and {len(overlay.events)-6} more events")

## Launch Demo Mode

Your DataPack is ready. Start MAIW Demo Mode:

**Quick setup (shell):**
```bash
./scripts/setup_demo_world.sh   # validates existing pack or generates new one
./scripts/start_demo_mode.sh    # starts API + UI
```

**Then open:** http://localhost:3001/demo

**Recommended scenario order:**
1. Run `healthy_baseline` — confirm world is coherent
2. Run `labor_constraint_wave_risk` — this is Scenario 001

**What you'll see:** OBSERVE → REASON → PROPOSE → DECIDE → APPROVE → EXECUTE → OUTCOME

---

**Your warehouse identity:**
```
Warehouse ID:   DC-47
Dataset ID:     dc47-demo-v1
Seed:           42
```

This identity appears in every trace_id, proposal, and outcome — letting you reproduce any run.

---

> **Note:** `stale_state` and `state_drift` currently use a compatibility adapter (healthy_baseline overlay).  
> Full migration is planned for a future phase.

> **Small preset:** Use `WarehouseWorldConfig.small()` for fast local validation without 25k SKUs.  
> The DC-47 preset is the canonical demo/evaluation world.

In [ ]:
from maiw_world.datapack import compute_semantic_checksum

print("=" * 60)
print("  MAIW v2 Setup Complete")
print("=" * 60)
print()
print(f"  DataPack: {PACK_DIR}")
print(f"  Checksum: {compute_semantic_checksum(g)[:32]}...")
print()
print("  Next: ./scripts/start_demo_mode.sh")
print("        http://localhost:3001/demo")